# The Anatomy of AI · Lecture 15 · More Ideas Than Neurons — the notebook

The lecture built the smallest network that has to hold more ideas than it has neurons, five ideas through two
neurons, trained it, and watched what it kept. Here you build and train it yourself:

1. the network, and what training asks of it;
2. importance and **sparsity** (how often an idea is absent);
3. training at three sparsities: two ideas kept, then four, then all five as a **pentagon** (*superposition*);
4. ten random starts, equal importance, and the same network without its gate;
5. why the gate makes sharing pay;
6. twenty ideas in five neurons;
7. what a neuron looks like now (*polysemantic*).

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab, CPU only, in about ten minutes; nothing to download, no keys.
**One difference from the lecture:** the lecture trained each model for 30,000 steps on batches of 4,096 samples (on a
GPU). Here the five-feature models train for 20,000 steps and the twenty-feature ones for 10,000 (the paper notebook's
number), on batches of 1,024, so that it all fits on Colab's CPU. The shapes come out the same; the losses come out a
little higher, and the checks allow for that.

## 0. Setup
Installs (pinned), the plot style, and `check()`.

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the small CPU build.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cpu")
pip("numpy==2.0.2", "matplotlib==3.10.0")

In [ ]:
import math, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch

T0 = time.time()
# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})
INKMAP = LinearSegmentedColormap.from_list("ink", [PAPER, INK])   # a value as ink density, as on the plates


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value) + 0.0:.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

## 1. Five ideas, two neurons (Plates I–III)
Five input numbers x₁…x₅, each between 0 and 1. Each is a **feature**: an idea the network may track. The network
squeezes them into two neurons and spreads them back out:

h = W x        (W has 2 × 5 weights: two neurons, five features)
x′ = ReLU(Wᵀ h + b)

The **gate** ReLU keeps positive numbers and turns negative ones to 0, as in Lecture 1. Training asks x′ to match x.
Read W by columns: feature i's two weights are a point in the plane of the two neurons, an **arrow**. Turn feature i on
alone (value 1) and the two neurons hold exactly its arrow; each output then measures that point along its own arrow
(a dot product, Lecture 14), adds its bias, and passes the gate.

In [ ]:
def forward(W, b, x, gate=True):
    """x: (batch, n) -> x' (batch, n). W: (m, n), b: (n,)."""
    h = x @ W.T                                       # squeeze: (batch, m)
    y = h @ W + b                                     # spread back out
    return np.maximum(y, 0) if gate else y


# A hand-made example: five arrows evenly round a circle, as a trained model will turn out to have.
ang = np.pi / 2 - np.arange(5) * 2 * np.pi / 5
W_hand = np.stack([np.cos(ang), np.sin(ang)])         # 2 x 5: column i is feature i's arrow
b_hand = np.full(5, -0.3)
x = np.zeros(5); x[0] = 1.0                            # feature 1 on, the rest off
h = W_hand @ x
print("the two neurons hold:", np.round(h, 3), "= feature 1's arrow:", np.round(W_hand[:, 0], 3))
print("each output reads, before the bias:", np.round(W_hand.T @ h, 3))
print("after bias and gate:", np.round(forward(W_hand, b_hand, x[None])[0], 3))
check("the neurons hold feature 1's arrow", np.allclose(h, W_hand[:, 0]), True)


def draw(ax, W, title="", scale=1.0):
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, ls=":", color=INK2))
    ax.axhline(0, color=INK2, lw=0.6); ax.axvline(0, color=INK2, lw=0.6)
    for i in range(W.shape[1]):
        L = np.hypot(*W[:, i])
        if L < 0.05:
            continue
        ax.annotate("", xy=W[:, i] * scale, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=INK if L > 0.5 else INK2, lw=2 if L > 0.5 else 1))
        ax.text(*(W[:, i] * scale * (1 + 0.22 / L)), str(i + 1), ha="center", va="center", fontsize=10)
    ax.set_xlim(-1.45, 1.45); ax.set_ylim(-1.45, 1.45); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)


fig, ax = plt.subplots(figsize=(3.2, 3.2)); draw(ax, W_hand, "five arrows in the plane of two neurons"); plt.show()

## 2. Importance and sparsity (Plate IV)
**Importance** is how much the network is paid for getting a feature right: feature 1 counts most, each next one 0.9
times as much (the paper notebook's choice). **Sparsity** S is how often a feature is 0. Each feature is 0 with
probability S, and otherwise a number drawn evenly between 0 and 1. The **loss** of a sample is Σᵢ importanceᵢ × (xᵢ − x′ᵢ)².

In [ ]:
IMP_DECAY = 0.9 ** np.arange(5)
print("importance:", np.round(IMP_DECAY, 2))
check("importance of feature 5", IMP_DECAY[4], 0.6561, 0.005)

gen = torch.Generator().manual_seed(0)
def sample(S, n_samples, n=5):
    on = torch.rand(n_samples, n, generator=gen) < 1 - S
    return (on * torch.rand(n_samples, n, generator=gen)).numpy()

for S in (0.0, 0.8, 0.9):
    X = sample(S, 200_000)
    print(f"S = {S}: share of feature values that are non-zero = {(X > 0).mean():.4f}")
check("at S = 0.9, a feature is present in about 1 sample in 10", (sample(0.9, 200_000) > 0).mean(), 0.1, 0.003)
fig, ax = plt.subplots(figsize=(8, 1.8))
ax.imshow(sample(0.9, 40).T, cmap=INKMAP, aspect="auto", vmin=0, vmax=1)
ax.set_yticks(range(5)); ax.set_yticklabels([f"feature {i + 1}" for i in range(5)], fontsize=8)
ax.set_xlabel("40 samples at S = 0.9"); plt.show()

## 3. Train it yourself (Plates V–IX)
Many small models train at once as one batched tensor: each row is one model with its own importance, sparsity, seed
and gate. Adam with a short warm-up and a cosine decay, as in the lecture. After training, each model's loss is measured
on 131,072 fresh samples, and for each setting we keep the best of 10 random starts, as the lecture (and the paper) did.

The rows: importance 0.9ⁱ at six sparsities (0, 0.8, 0.84, 0.9, 0.95, 0.99); equal importance at 0.9; and the same
network **without the gate** (x′ = Wᵀh + b) at 0 and 0.9. Ten random starts each: 90 models, 20,000 steps. It takes
about five minutes.

In [ ]:
def train(rows, n, m, steps=10_000, batch=1024, lr=1e-3, seed=0, snaps=()):
    """rows: list of (importance vector, sparsity S, gate?). Trains all rows at once; returns final W, b, losses and
    the weights at the snapshot steps."""
    g = torch.Generator().manual_seed(seed)
    R = len(rows)
    I = torch.tensor(np.array([r[0] for r in rows]), dtype=torch.float32)
    P = torch.tensor([[1 - r[1]] * n for r in rows], dtype=torch.float32)       # probability a feature is present
    gate = torch.tensor([r[2] for r in rows])
    W = torch.empty(R, m, n)
    for r in range(R):
        torch.nn.init.xavier_normal_(W[r], generator=g)
    W.requires_grad_(True)
    b = torch.zeros(R, n, requires_grad=True)
    opt = torch.optim.Adam([W, b], lr=lr)
    warm = int(0.05 * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * s / steps))))

    def batch_of(B):
        on = torch.rand(R, B, n, generator=g) < P[:, None, :]
        return on * torch.rand(R, B, n, generator=g)

    def out(x):
        h = torch.einsum("rbn,rmn->rbm", x, W)
        y = torch.einsum("rbm,rmn->rbn", h, W) + b[:, None, :]
        return torch.where(gate[:, None, None], torch.relu(y), y)

    snaps, saved = sorted(snaps), []
    for step in range(steps + 1):
        if step in snaps:
            saved.append(W.detach().clone().numpy())
        if step == steps:
            break
        x = batch_of(batch)
        loss = (I[:, None, :] * (x - out(x)) ** 2).sum(-1).mean(-1)
        opt.zero_grad(set_to_none=True); loss.sum().backward(); opt.step(); sched.step()
    with torch.no_grad():
        final = sum((I[:, None, :] * (x - out(x)) ** 2).sum(-1).mean(-1) for x in (batch_of(8192) for _ in range(16))) / 16
    return W.detach().numpy(), b.detach().numpy(), final.numpy(), np.array(saved)


def norms(W):
    return np.hypot(W[0], W[1])


def kept(W):
    return int((norms(W) > 0.5).sum())


def gaps(W):
    """Angles (degrees) between neighbouring kept arrows, going round the circle."""
    a_ = np.sort(np.degrees(np.arctan2(W[1], W[0]))[norms(W) > 0.5])
    return np.diff(np.concatenate([a_, [a_[0] + 360]])) if len(a_) > 1 else np.array([])


SEEDS = 10
SPARS = [0.0, 0.8, 0.8415, 0.9, 0.95, 0.99]
groups = [("decay", S, IMP_DECAY, True) for S in SPARS] + [("equal", 0.9, np.ones(5), True)] + \
         [("no gate", S, IMP_DECAY, False) for S in (0.0, 0.9)]
rows = [(imp, S, gate) for _, S, imp, gate in groups for _ in range(SEEDS)]
SNAPS = sorted(set(np.unique(np.round(np.logspace(0, np.log10(20_000), 40)).astype(int)).tolist() + [0]))
t0 = time.time()
W_all, b_all, loss_all, snap_all = train(rows, n=5, m=2, steps=20_000, seed=1, snaps=SNAPS)
print(f"trained {len(rows)} models in {time.time() - t0:.0f} s")

best = {}
for k, (name, S, _, _) in enumerate(groups):
    sl = slice(k * SEEDS, (k + 1) * SEEDS)
    i = k * SEEDS + int(np.argmin(loss_all[sl]))
    best[(name, S)] = dict(W=W_all[i], b=b_all[i], loss=float(loss_all[i]), row=i,
                           kept_seeds=[kept(W_all[j]) for j in range(k * SEEDS, (k + 1) * SEEDS)])

**Dense features, S = 0 (Plate V).** Every feature is present in every sample. The lecture's network kept the two most
important, at right angles, and gave up on the other three: their arrows shrink to nothing and their biases settle at
0.5, their average value.

In [ ]:
B0 = best[("decay", 0.0)]
print("lengths:", np.round(norms(B0["W"]), 3), " biases:", np.round(B0["b"], 3))
check("S = 0: features kept", kept(B0["W"]), 2)
check("S = 0: the kept ones are features 1 and 2", sorted(np.where(norms(B0["W"]) > 0.5)[0].tolist()), [0, 1])
check("S = 0: angle between them", gaps(B0["W"])[0] if gaps(B0["W"])[0] <= 180 else 360 - gaps(B0["W"])[0], 90.0, 1.0)
check("S = 0: the dropped features' biases (their average value)", float(B0["b"][2:].mean()), 0.5, 0.02)
check("S = 0: loss", B0["loss"], 0.18245, 0.05 * 0.18245)

**The sparsity slider (Plate VI).** Make the features rarer and train again. At S = 0.8 the lecture's network kept four,
as two opposite pairs; at S = 0.9 all five, as a pentagon, with negative biases. Five ideas in two neurons, each on its
own arrow: **superposition**.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, S in zip(axs, (0.0, 0.8, 0.9)):
    Bk = best[("decay", S)]
    draw(ax, Bk["W"], f"S = {S}: {kept(Bk['W'])} kept")
plt.show()
B8, B9 = best[("decay", 0.8)], best[("decay", 0.9)]
print("S = 0.8 gaps:", np.round(gaps(B8["W"]), 1), " S = 0.9 gaps:", np.round(gaps(B9["W"]), 1), " biases:", np.round(B9["b"], 3))
check("S = 0.8: features kept", kept(B8["W"]), 4)
check("S = 0.8: the one dropped is the least important, feature 5", int(np.argmin(norms(B8["W"]))), 4)
check("S = 0.8: the four sit at right angles (largest gap error)", float(np.abs(gaps(B8["W"]) - 90).max()), 0.0, 3.0)
check("S = 0.9: features kept", kept(B9["W"]), 5)
check("S = 0.9: a pentagon (largest gap error from 72 degrees)", float(np.abs(gaps(B9["W"]) - 72).max()), 0.0, 5.0)
check("S = 0.9: mean bias", float(B9["b"].mean()), -0.206, 0.03)
check("S = 0.9: loss", B9["loss"], 0.02135, 0.1 * 0.02135)

**Watch it form.** We saved the S = 0.9 models' weights at 40 steps during training. Here is the best one forming.

In [ ]:
fig, axs = plt.subplots(1, 6, figsize=(12, 2.3))
for ax, k in zip(axs, np.linspace(0, len(SNAPS) - 1, 6).astype(int)):
    draw(ax, snap_all[k][B9["row"]], f"step {SNAPS[k]:,}")
plt.show()

**Ten random starts (Plate VII).** Near the switch from four to five (S = 0.84), the lecture's ten starts disagreed:
some kept 3, some 4, some 5. Gradient descent can settle in a worse layout, which is why we keep the best of ten.

In [ ]:
print("S = 0.84, features kept by each of the 10 starts:", best[("decay", 0.8415)]["kept_seeds"])
print("S = 0.95:", best[("decay", 0.95)]["kept_seeds"])
check("S = 0.84: the ten starts do not all agree", len(set(best[("decay", 0.8415)]["kept_seeds"])) > 1, True)
check("S = 0.84: the best of ten keeps", kept(best[("decay", 0.8415)]["W"]), 5)
check("S = 0.95: the best of ten keeps", kept(best[("decay", 0.95)]["W"]), 5)

**Equal importance (Plate VIII).** Pay the network the same for every feature, and the pentagon comes out regular.

In [ ]:
E9 = best[("equal", 0.9)]
print("gaps:", np.round(gaps(E9["W"]), 1), " lengths:", np.round(norms(E9["W"]), 3), " biases:", np.round(E9["b"], 3))
check("equal, S = 0.9: features kept", kept(E9["W"]), 5)
check("equal, S = 0.9: largest gap error from 72 degrees", float(np.abs(gaps(E9["W"]) - 72).max()), 0.0, 1.5)
check("equal, S = 0.9: arrow length", float(norms(E9["W"]).mean()), 1.1, 0.03)
check("equal, S = 0.9: bias", float(E9["b"].mean()), -0.209, 0.02)
check("equal, S = 0.9: loss", E9["loss"], 0.0258, 0.1 * 0.0258)
fig, ax = plt.subplots(figsize=(3.2, 3.2)); draw(ax, E9["W"], "equal importance, S = 0.9"); plt.show()

**Without the gate (Plate IX).** Take ReLU away: x′ = Wᵀh + b. The lecture's linear network kept exactly two features at
every sparsity, and lost about three times as much at S = 0.9.

In [ ]:
L0, L9 = best[("no gate", 0.0)], best[("no gate", 0.9)]
check("no gate, S = 0: features kept", kept(L0["W"]), 2)
check("no gate, S = 0.9: features kept", kept(L9["W"]), 2)
check("no gate, S = 0.9: total squared length of W", float((L9["W"] ** 2).sum()), 2.0, 0.05)
check("no gate, S = 0.9: loss", L9["loss"], 0.06729, 0.05 * 0.06729)
print(f"loss without the gate / with it, at S = 0.9: {L9['loss'] / B9['loss']:.1f}")
check("without the gate costs about this many times more", L9["loss"] / B9["loss"], 3.15, 0.5)

## 4. Why the gate makes sharing pay (Plate X)
Take the regular pentagon and turn one feature on at value 1. Each output reads it along its own arrow and adds its
bias. Its own output comes back at about 1; its two **neighbours** (72° away) hear a small false echo; the two **far** ones
read below zero, and the gate clips them away. The echo survives only when the feature is stronger than a threshold.

In [ ]:
W_, b_ = E9["W"], E9["b"]
Gm = W_.T @ W_                                         # every arrow measured along every arrow
a_ = 3                                                 # feature 4 on, as in the lecture
others = sorted((j for j in range(5) if j != a_), key=lambda j: -Gm[a_, j])
nbr, far_ = others[:2], others[2:]
x = np.zeros(5); x[a_] = 1
pre = Gm[:, a_] + b_
print("before the gate:", np.round(pre, 3), "\nafter it:      ", np.round(np.maximum(pre, 0), 3))
check("its own reading along its arrow", Gm[a_, a_], 1.21, 0.05)
check("a neighbour's reading", float(Gm[a_, nbr].mean()), 0.373, 0.03)
check("a far one's reading", float(Gm[a_, far_].mean()), -0.98, 0.05)
check("its own output after bias and gate", float(np.maximum(pre[a_], 0)), 1.001, 0.05)
thr = float(-b_[nbr].mean() / Gm[a_, nbr].mean())
check("the echo survives only above this value", thr, 0.559, 0.05)
x2 = np.zeros(5); x2[a_] = x2[nbr[0]] = 1
two = forward(W_, b_, x2[None])[0]
check("two neighbours on together: each reads", float(two[[a_, nbr[0]]].mean()), 1.374, 0.05)

## 5. Twenty ideas, five neurons (Plate XI)
The paper's bigger test: 20 features, 5 neurons, importance 0.7ⁱ. A feature is **kept** if its arrow is longer than 0.5,
and **in superposition** if it also overlaps the others: its overlaps with every other arrow, measured along its own
direction, squared and summed, exceed 0.1. We train four starts at three sparsities, and the same without the gate.
The lecture trained eight starts for longer and they did not all agree, so each check below allows the spread of
the lecture's own eight.

In [ ]:
IMP20 = 0.7 ** np.arange(20)
groups20 = [("gate", S, True) for S in (0.0, 0.9, 0.99)] + [("no gate", 0.99, False)]
rows20 = [(IMP20, S, gate) for _, S, gate in groups20 for _ in range(4)]
t0 = time.time()
W20, b20, l20, _ = train(rows20, n=20, m=5, seed=2)
print(f"trained {len(rows20)} models in {time.time() - t0:.0f} s")


def census(W):
    nr = np.linalg.norm(W, axis=0)
    Wh = W / np.maximum(nr, 1e-9)
    Gh = Wh.T @ W
    inter = (Gh ** 2).sum(1) - np.diag(Gh) ** 2
    keep = nr > 0.5
    return int(keep.sum()), int((keep & (inter > 0.1)).sum())


res20 = {}
for k, (name, S, _) in enumerate(groups20):
    i = 4 * k + int(np.argmin(l20[4 * k:4 * k + 4]))
    res20[(name, S)] = (census(W20[i]), W20[i])
    print(f"{name:8s} S = {S}: kept {res20[(name, S)][0][0]:2d}, in superposition {res20[(name, S)][0][1]:2d}")
check("S = 0: kept", res20[("gate", 0.0)][0][0], 5)
check("S = 0: in superposition", res20[("gate", 0.0)][0][1], 0)
# the lecture's eight starts did not all agree either; its best start is the expected value, its spread the tolerance
print("the lecture's eight starts, (kept, in superposition): S = 0.9", [[9, 8], [9, 8], [9, 8], [9, 8], [9, 8], [9, 8], [10, 10], [10, 10]], "| S = 0.99", [[10, 10], [11, 11], [11, 11], [11, 11], [12, 12], [12, 12], [12, 12], [13, 13]])
check("S = 0.9: kept", res20[("gate", 0.9)][0][0], 9, 1)
check("S = 0.9: in superposition", res20[("gate", 0.9)][0][1], 8, 2)
check("S = 0.99: kept", res20[("gate", 0.99)][0][0], 13, 3)
check("S = 0.99: in superposition", res20[("gate", 0.99)][0][1], 13, 3)
check("no gate, S = 0.99: kept", res20[("no gate", 0.99)][0][0], 5)

fig, axs = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, S in zip(axs, (0.0, 0.9, 0.99)):
    Wk = res20[("gate", S)][1]
    ax.imshow(Wk.T @ Wk, cmap="gray_r", vmin=-1, vmax=1)
    ax.set_title(f"WᵀW at S = {S}", fontsize=10); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 6. What a neuron looks like now (Plate XII)
Ask one neuron what it answers to, as in Lecture 6: turn each feature on alone and read the neuron. A reading counts if
it is over 0.3 either way. The toy's two neurons have no gate of their own, so we may turn both axes together without
changing anything the network computes; we try every turn and keep the best.

In [ ]:
def busiest(W, thr=0.3):
    """The fewest features the busier neuron must answer to, over every turn of the two axes."""
    best_ = 99
    for t in np.radians(np.arange(0, 180, 0.5)):
        R_ = np.array([[np.cos(t), np.sin(t)], [-np.sin(t), np.cos(t)]])
        best_ = min(best_, int((np.abs(R_ @ W) > thr).sum(1).max()))
    return best_


print("as trained, each neuron answers to", (np.abs(B9["W"]) > 0.3).sum(1), "of the 5 features")
check("S = 0: whatever the turn, the busier neuron needs at most", busiest(B0["W"]), 1)
check("S = 0.9: whatever the turn, the busier neuron answers to at least", busiest(B9["W"]), 4)

With rare features, each neuron answers to several: **polysemantic**, not because it is confused, but because five
ideas share two neurons.

## Your turn
1. **Rarer still.** Look at the S = 0.99 model (already trained). How many features does it keep, and are its biases more
   negative than at 0.9? *Hint:* `best[("decay", 0.99)]`.
2. **Who goes first?** At S = 0.8 one feature was dropped. Why that one? *Hint:* compare the importances.
3. **How rare is a collision?** At S = 0.9, how often are two given features on together? Count it in 200,000 samples.
4. **A stricter threshold.** Redo Section 6 with a reading counting only if over 0.5. *Hint:* `busiest(W, thr=0.5)`.
5. **Twenty, which ones?** At S = 0.9 in the twenty-feature model, which features are kept? *Hint:* the arrow lengths.

In [ ]:
# TODO 1

In [ ]:
# TODO 2

In [ ]:
# TODO 3

In [ ]:
# TODO 4

In [ ]:
# TODO 5

## Solutions

In [ ]:
# 1. Still five, and the bias is more negative: rarer features need a stronger filter against the echo.
B99 = best[("decay", 0.99)]
print("S = 0.99 lengths:", np.round(norms(B99["W"]), 2), " mean bias:", round(float(B99["b"].mean()), 3))
check("S = 0.99: features kept", kept(B99["W"]), 5)
check("S = 0.99: mean bias", float(B99["b"].mean()), -0.274, 0.04)
check("the bias is more negative than at S = 0.9", float(B99["b"].mean()) < float(B9["b"].mean()), True)

In [ ]:
# 2. Feature 5 has the smallest importance (0.66): dropping it costs the least.
print("importances:", np.round(IMP_DECAY, 2), " lengths at S = 0.8:", np.round(norms(B8["W"]), 2))
check("the dropped feature has the smallest importance", int(np.argmin(norms(B8["W"]))), int(np.argmin(IMP_DECAY)))

In [ ]:
# 3. Each is on 1 time in 10, so both together about 1 time in 100.
X = sample(0.9, 200_000)
both = float(((X[:, 0] > 0) & (X[:, 1] > 0)).mean())
print(f"features 1 and 2 on together in {both:.4f} of samples")
check("two given features on together at S = 0.9", both, 0.01, 0.001)

In [ ]:
# 4. With the stricter threshold the counts drop, but still no turn gives each feature a neuron of its own.
print("best turn, threshold 0.5:", busiest(B9["W"], thr=0.5))
check("no turn gives each feature its own neuron (busier neuron answers to 2 or more)", busiest(B9["W"], thr=0.5) >= 2, True)

In [ ]:
# 5. The most important ones are kept; the least important are dropped first.
nr9 = np.linalg.norm(res20[("gate", 0.9)][1], axis=0)
print("kept at S = 0.9:", (np.where(nr9 > 0.5)[0] + 1).tolist())
check("feature 1 (the most important) is kept", bool(nr9[0] > 0.5), True)
check("feature 20 (the least important) is dropped", bool(nr9[19] <= 0.5), True)

In [ ]:
print(f"notebook finished in {(time.time() - T0) / 60:.1f} min")